In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from cellassign import assign_cats

from cellbender.remove_background.downstream import load_anndata_from_input_and_output as load_anndata_cellbender

import numpy as np

import os

import pandas as pd

import plotly.express as px

from pybiomart import Server

import re 

from rpy2.robjects import globalenv
from rpy2.robjects import pandas2ri

import scanpy as sc
import scanpy.external as sce

import scFates as scf

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

import scipy.sparse as sp
from scipy.sparse import csr_matrix, issparse

import scvelo as scv

import seaborn as sns

from sklearn.decomposition import PCA

import triku as tk

In [ ]:
import gseapy as gp

In [ ]:
from pyfuncs.general import preprocessing_adata_sub, set_plotting_style, plot_volcano
from pyfuncs.general import magma

set_plotting_style()

In [ ]:
os.makedirs('results/ARAUZO_03', exist_ok=True)

---

In [ ]:
adata = sc.read_h5ad('data/ARAUZO_03/processed_adatas/adata_processed.h5ad')

In [ ]:
sc.pl.umap(adata, color=['major_population', 'minor_population'], ncols=1, cmap=magma)

# Separating FAPs

In [ ]:
adata_FAP = adata[adata.obs['major_population'] == 'FAP'].copy()
preprocessing_adata_sub(adata_FAP, integrate = True, n_comps=30, k=90)

In [ ]:
sc.tl.umap(adata_FAP, min_dist=0.5)

In [ ]:
sc.pl.umap(adata_FAP, color=['minor_population'])

In [ ]:
# Classical Lum/Prg4 FAP division
sc.pl.umap(adata_FAP, color=['Lum', 'Prg4'], cmap=magma, use_raw=False)


# The division is not very good, and does not respond to this pattern.

## 2-tier FAP division

In [ ]:
sc.tl.leiden(adata_FAP, resolution=0.07, key_added='leiden_FAP_0.07')
sc.pl.umap(adata_FAP, color=['leiden_FAP_0.07'])

In [ ]:
sc.tl.rank_genes_groups(adata_FAP, groupby='leiden_FAP_0.07', method='wilcoxon', use_raw=False)
df_DEGs_tier_2_A = plot_volcano(adata_FAP, cluster='0', xlim=[0, 10], plot_positive_only=False, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=1, pval_threshold=1e-50)
df_DEGs_tier_2_BCDEF = plot_volcano(adata_FAP, cluster='1', xlim=[0, 10], plot_positive_only=False, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=1, pval_threshold=1e-50)

In [ ]:
dict_2_tier = {
    'FAP_A': ['Igfbp7', 'Smoc2', 'Col15a1', 'Bgn', 'Mgp', 'Crispld2', 'Col4a2',
       'Col4a1', 'Lum', 'Cxcl14', 'Cygb', 'Sparcl1', 'Cst3', 'Svep1',
       'Nfib', 'Gas6', 'Ank2', 'Hmcn2', 'Gas1', 'Ltbp4'],
    'FAP_BCDEF': ['Mfap5', 'Ugdh', 'Opcml', 'Adgrd1', 'Sdk1', 'Cd55', 'Gfpt2',
       'Pcolce2', 'Emilin2', 'Limch1', 'Fbn1', 'Creb5', 'Fn1', 'Cd248',
       'Efhd1', 'Anxa3', 'Dpp4', 'Pi16', 'Efemp1', 'Sema3c'],
}

In [ ]:
assign_cats(adata_FAP, dict_2_tier, key_added='cats_2tier', column_groupby='leiden_FAP_0.07')

sc.pl.umap(adata_FAP, color=['cats_2tier'])

In [ ]:
df_DEGs_tier_2_A.to_csv('results/ARAUZO_03/DEGs_tier_2_A.csv', index=False)
df_DEGs_tier_2_BCDEF.to_csv('results/ARAUZO_03/DEGs_tier_2_BCDEF.csv', index=False)

### GO analysis

In [ ]:
DEGs_A = df_DEGs_tier_2_A.iloc[:100, :]['gene'].values.tolist()
DEGs_BCDEF = df_DEGs_tier_2_BCDEF.iloc[:100, :]['gene'].values.tolist()

In [ ]:
enr_tier_2_A = gp.enrichr(
        gene_list=DEGs_A,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_2_A.results.to_csv('results/ARAUZO_03/enr_tier_2_A.csv', index=False)

enr_tier_2_A.results

In [ ]:
enr_tier_2_BCDEF = gp.enrichr(
        gene_list=DEGs_BCDEF,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_2_BCDEF.results.to_csv('results/ARAUZO_03/enr_tier_2_BCDEF.csv', index=False)

enr_tier_2_BCDEF.results

## 3-tier FAP division

In [ ]:
sc.tl.leiden(adata_FAP, resolution=0.15, key_added='leiden_FAP_0.15')
sc.pl.umap(adata_FAP, color=['leiden_FAP_0.15'])

In [ ]:
sc.tl.rank_genes_groups(adata_FAP, groupby='leiden_FAP_0.15', method='wilcoxon', use_raw=False)
df_DEGs_tier_3_A = plot_volcano(adata_FAP, cluster='0', xlim=[0,  10], plot_positive_only=True, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=2, pval_threshold=1e-50)
df_DEGs_tier_3_B = plot_volcano(adata_FAP, cluster='2', xlim=[0,  10], plot_positive_only=False, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=1, pval_threshold=1e-50)
df_DEGs_tier_3_CDEF = plot_volcano(adata_FAP, cluster='1', xlim=[0,  10], plot_positive_only=False, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=1, pval_threshold=1e-50)

In [ ]:
df_DEGs_tier_3_CDEF.iloc[:20, :]['gene'].values

In [ ]:
dict_3_tier = {
    'FAP_A': ['Sema3c', 'Efemp1', 'Pi16', 'Anxa3', 'Dpp4', 'Efhd1', 'Cd248',
       'Fn1', 'Creb5', 'Limch1', 'Fbn1', 'Emilin2', 'Gfpt2', 'Pcolce2',
       'Opcml', 'Cd55', 'Adgrd1', 'Sdk1', 'Ugdh', 'Mfap5'],
    'FAP_B': ['Smoc2', 'Col4a1', 'Col15a1', 'Hspg2', 'Hmcn2', 'Lamb1', 'Ccl11',
       'Crispld2', 'Col4a2', 'Hsd11b1', 'Enpp2', 'Ret', 'Grm8', 'Cxcl14',
       'Mme', 'Col5a3', 'Col6a2', 'Angptl1', 'Abca8a', 'Lamc1'],
    'FAP_CDEF': ['Cst3', 'Gas6', 'Mgp', 'Igfbp7', 'Bgn', 'Gdf10', 'Gas1', 'Apod',
       'Cygb', 'Cfh', 'Srpx', 'Il11ra1', 'Fbln1', 'Boc', 'Nrp1', 'Nfib',
       'Olfml3', 'Penk', 'Fgl2', 'Prg4']
}

In [ ]:
assign_cats(adata_FAP, dict_3_tier, key_added='cats_3tier', column_groupby='leiden_FAP_0.15')

sc.pl.umap(adata_FAP, color=['cats_3tier'])

In [ ]:
df_DEGs_tier_3_A.to_csv('results/ARAUZO_03/DEGs_tier_3_A.csv', index=False)
df_DEGs_tier_3_B.to_csv('results/ARAUZO_03/DEGs_tier_3_B.csv', index=False)
df_DEGs_tier_3_CDEF.to_csv('results/ARAUZO_03/DEGs_tier_3_CDEF.csv', index=False)

### GO analysis

In [ ]:
DEGs_A = df_DEGs_tier_3_A.iloc[:100, :]['gene'].values.tolist()
DEGs_B = df_DEGs_tier_3_B.iloc[:100, :]['gene'].values.tolist()
DEGs_CDEF = df_DEGs_tier_3_CDEF.iloc[:100, :]['gene'].values.tolist()

In [ ]:
enr_tier_3_A = gp.enrichr(
        gene_list=DEGs_A,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_3_A.results.to_csv('results/ARAUZO_03/enr_tier_3_A.csv', index=False)

enr_tier_3_A.results

In [ ]:
enr_tier_3_B = gp.enrichr(
        gene_list=DEGs_B,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_3_B.results.to_csv('results/ARAUZO_03/enr_tier_3_B.csv', index=False)

enr_tier_3_B.results

In [ ]:
enr_tier_3_CDEF = gp.enrichr(
        gene_list=DEGs_CDEF,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_3_CDEF.results.to_csv('results/ARAUZO_03/enr_tier_3_CDEF.csv', index=False)

enr_tier_3_CDEF.results

## 4-tier FAP division

In [ ]:
sc.tl.leiden(adata_FAP, resolution=0.25, key_added='leiden_FAP_0.25')
sc.pl.umap(adata_FAP, color=['leiden_FAP_0.25'])

In [ ]:
sc.tl.rank_genes_groups(adata_FAP, groupby='leiden_FAP_0.25', method='wilcoxon', use_raw=False)
df_DEGs_tier_4_A = plot_volcano(adata_FAP, cluster='0', xlim=[0,  10], plot_positive_only=True, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=2, pval_threshold=1e-50)
df_DEGs_tier_4_B = plot_volcano(adata_FAP, cluster='1', xlim=[0,  10], plot_positive_only=False, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=1, pval_threshold=1e-50)
df_DEGs_tier_4_E = plot_volcano(adata_FAP, cluster='2', xlim=[0,  10], plot_positive_only=False, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=1, pval_threshold=1e-50)
df_DEGs_tier_4_CDF = plot_volcano(adata_FAP, cluster='3', xlim=[0,  10], plot_positive_only=False, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=1, pval_threshold=1e-50)

In [ ]:
df_DEGs_tier_4_CDF.iloc[:20, :]['gene'].values

In [ ]:
dict_4_tier = {
    'FAP_A': ['Sema3c', 'Efemp1', 'Dpp4', 'Anxa3', 'Pi16', 'Efhd1', 'Cd248',
       'Creb5', 'Fn1', 'Fbn1', 'Limch1', 'Emilin2', 'Cd55', 'Pcolce2',
       'Gfpt2', 'Sdk1', 'Adgrd1', 'Ugdh', 'Mfap5', 'Opcml'],
    'FAP_B': ['Smoc2', 'Col4a1', 'Col15a1', 'Hmcn2', 'Hspg2', 'Lamb1', 'Hsd11b1',
       'Ccl11', 'Crispld2', 'Col4a2', 'Cxcl14', 'Grm8', 'Mme', 'Angptl1',
       'Enpp2', 'Ret', 'Col6a2', 'Col5a3', 'Abca8a', 'Lamc1'],
    'FAP_E': ['Fbln1', 'Srpx', 'Il11ra1', 'Gdf10', 'Col1a2', 'Gas6', 'Igfbp7',
       'Ccdc80', 'C7', 'Col1a1', 'Gas1', 'Serpinf1', 'Olfml3', 'Sfrp1',
       'Apod', 'Cst3', 'C2', 'Igf1', 'Gpnmb', 'Bgn'],
    'FAP_CDF': ['Mgp', 'Hmcn1', 'Meox2', 'Prg4', 'Cst3', 'Boc', 'Colec12', 'Myoc',
       'Meox1', 'Emp1', 'Sparcl1', 'Kctd12', 'Cfh', 'Sptbn1', 'Cd9',
       'Aopep', 'Etl4', 'Serpine2', 'Trps1', 'Matn2']}

In [ ]:
assign_cats(adata_FAP, dict_4_tier, key_added='cats_4tier', column_groupby='leiden_FAP_0.25')

sc.pl.umap(adata_FAP, color=['cats_4tier'])

In [ ]:
df_DEGs_tier_4_A.to_csv('results/ARAUZO_03/DEGs_tier_4_A.csv', index=False)
df_DEGs_tier_4_B.to_csv('results/ARAUZO_03/DEGs_tier_4_B.csv', index=False)
df_DEGs_tier_4_E.to_csv('results/ARAUZO_03/DEGs_tier_4_E.csv', index=False)
df_DEGs_tier_4_CDF.to_csv('results/ARAUZO_03/DEGs_tier_4_CDF.csv', index=False)

### GO analysis

In [ ]:
DEGs_A = df_DEGs_tier_4_A.iloc[:100, :]['gene'].values.tolist()
DEGs_B = df_DEGs_tier_4_B.iloc[:100, :]['gene'].values.tolist()
DEGs_E = df_DEGs_tier_4_E.iloc[:100, :]['gene'].values.tolist()
DEGs_CDF = df_DEGs_tier_4_CDF.iloc[:100, :]['gene'].values.tolist()

In [ ]:
enr_tier_4_A = gp.enrichr(
        gene_list=DEGs_A,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_4_A.results.to_csv('results/ARAUZO_03/enr_tier_4_A.csv', index=False)

enr_tier_4_A.results

In [ ]:
enr_tier_4_B = gp.enrichr(
        gene_list=DEGs_B,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_4_B.results.to_csv('results/ARAUZO_03/enr_tier_4_B.csv', index=False)

enr_tier_4_B.results

In [ ]:
enr_tier_4_E = gp.enrichr(
        gene_list=DEGs_E,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_4_E.results.to_csv('results/ARAUZO_03/enr_tier_4_E.csv', index=False)

enr_tier_4_E.results

In [ ]:
enr_tier_4_CDF = gp.enrichr(
        gene_list=DEGs_CDF,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_4_CDF.results.to_csv('results/ARAUZO_03/enr_tier_4_CDF.csv', index=False)

enr_tier_4_CDF.results

## 6-tier FAP division (FAPs A to F)

In [ ]:
dict_6_FAP = {
    'FAP_A': ['Sbsn', 'Efna5', 'Adamts16', 'Dact2', 'Krtdap', 'Aldh1a3', 'Ildr2', 
              'Car8', 'Rorb', 'Uchl1', 'Bmp3', 'Sema3e', 'Dmkn', 'Ano3', 'Itgb7'] + \
             ['Cmah', 'Stmn4', 'Robo1', 'Fam167a', 'Duox1', 'Gm33624', 
               'Cd55', 'Calm1', 'Ugp2',  'Prdm8', 'Kcp', 'Des'] + \
            ['C3', 'Postn', 'Slc4a4', 'Ifi27l2a', 'Il1rl2', 'Smpd3', 
               'Il1r2', 'Sfrp2', 'Vegfd', 'Cxcl13', 'Prkg2', 'C7', 
               'Cpz', 'Cdh9', 'Edn1', 'Sema3b', 'Mmp27'],
    
    'FAP_B': ['Grm8', 'Rora', 'Plxdc2', 'Ank2', 'Pcdh7', 'Eda', 'Tnik', 'Adgrl3', 'Elmo1'] +  \
             ['Cxcl14', 'Hsd11b1', 'Smoc2', 'Col15a1', 'Mme', 'G0s2', 'Col4a1', 'Lamb1', 'Col4a2', 
               'Cldn15', 'Clec14a', 'Nmb', 'Vwa1', 'Crlf1',],
    'FAP_C': ['Gdf10', 'C2', 'Cygb', 'Fmo2', 'Nr2f2', 'Csmd1', 'Prdm6', 'Abcc9', 'Gria4', 'Clec11a', 'Ism1', 
               'Gata6', 'Pltp', 'Hhip', 'Tspan11', 'Tent5c'],
    'FAP_D': ['Steap4', 'Apoe', 'Cyp1b1', 'Aoc3', 'Ccl19-ps4', 'Sfrp1', 'Adam12', 'Emb', 
               'Agt', 'Pparg',  ],
    'FAP_E': ['Mgp', 'Hmcn1', 'Meox1', 'Meox2', 'Clu', 'Etl4', 'Kctd12', 'Boc', 'Daam2', 'Matn2', 
              'Robo2', 'Clec1a', 'Myo10', 'Sgip1', 'Myo1b', 'Ptn', 'Mettl24', 'Rgs7bp'],
    'FAP_F': ['Thbs2', 'Gpx3', 'Cilp', 'Fgl2', 'Lox', 'Aspn', 'Ccdc80', 'Igfbp3', 'Dkk2', 'Ecrg4', 'Tnmd', 
              'Col12a1', 'Pappa2', 'Egfl6', 'Fibin', 'Ccn5', 'Sfrp4', 'Fxyd6', ],
    }

In [ ]:
sc.tl.leiden(adata_FAP, resolution=1, key_added='leiden_FAP_1')
sc.pl.umap(adata_FAP, color=['leiden_FAP_1'])

In [ ]:
assign_cats(adata_FAP, dict_6_FAP, key_added='cats_6tier', column_groupby='leiden_FAP_1')

sc.pl.umap(adata_FAP, color=['cats_6tier'])

In [ ]:
sc.tl.rank_genes_groups(adata_FAP, groupby='cats_6tier', method='wilcoxon', use_raw=False)

df_DEGs_tier_6_A = plot_volcano(adata_FAP, cluster='FAP_A', xlim=[0,  10], plot_positive_only=True, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=0.8, pval_threshold=1e-50)
df_DEGs_tier_6_B = plot_volcano(adata_FAP, cluster='FAP_B', xlim=[0,  10], plot_positive_only=True, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=0.8, pval_threshold=1e-50)
df_DEGs_tier_6_C = plot_volcano(adata_FAP, cluster='FAP_C', xlim=[0,  10], plot_positive_only=True, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=0.8, pval_threshold=1e-25)
df_DEGs_tier_6_D = plot_volcano(adata_FAP, cluster='FAP_D', xlim=[0,  10], plot_positive_only=True, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=0.8, pval_threshold=1e-25)
df_DEGs_tier_6_E = plot_volcano(adata_FAP, cluster='FAP_E', xlim=[0,  10], plot_positive_only=True, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=0.8, pval_threshold=1e-25)
df_DEGs_tier_6_F = plot_volcano(adata_FAP, cluster='FAP_F', xlim=[0,  10], plot_positive_only=True, 
            topn=20, bottomn=0, return_df=True, lfc_threshold=0.8, pval_threshold=1e-25)

In [ ]:
df_DEGs_tier_6_A.to_csv('results/ARAUZO_03/DEGs_tier_6_A.csv', index=False)
df_DEGs_tier_6_B.to_csv('results/ARAUZO_03/DEGs_tier_6_B.csv', index=False)
df_DEGs_tier_6_C.to_csv('results/ARAUZO_03/DEGs_tier_6_C.csv', index=False)
df_DEGs_tier_6_D.to_csv('results/ARAUZO_03/DEGs_tier_6_D.csv', index=False)
df_DEGs_tier_6_E.to_csv('results/ARAUZO_03/DEGs_tier_6_E.csv', index=False)
df_DEGs_tier_6_F.to_csv('results/ARAUZO_03/DEGs_tier_6_F.csv', index=False)

### GO analysis

In [ ]:
DEGs_A = df_DEGs_tier_6_A.iloc[:100, :]['gene'].values.tolist()
DEGs_B = df_DEGs_tier_6_B.iloc[:100, :]['gene'].values.tolist()
DEGs_C = df_DEGs_tier_6_C.iloc[:100, :]['gene'].values.tolist()
DEGs_D = df_DEGs_tier_6_D.iloc[:100, :]['gene'].values.tolist()
DEGs_E = df_DEGs_tier_6_E.iloc[:100, :]['gene'].values.tolist()
DEGs_F = df_DEGs_tier_6_F.iloc[:100, :]['gene'].values.tolist()


In [ ]:
enr_tier_6_A = gp.enrichr(
        gene_list=DEGs_A,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_6_A.results.to_csv('results/ARAUZO_03/enr_tier_6_A.csv', index=False)

enr_tier_6_A.results

In [ ]:
enr_tier_6_B = gp.enrichr(
        gene_list=DEGs_B,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_6_B.results.to_csv('results/ARAUZO_03/enr_tier_6_B.csv', index=False)

enr_tier_6_B.results

In [ ]:
enr_tier_6_C = gp.enrichr(
        gene_list=DEGs_C,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_6_C.results.to_csv('results/ARAUZO_03/enr_tier_6_C.csv', index=False)

enr_tier_6_C.results

In [ ]:
enr_tier_6_D = gp.enrichr(
        gene_list=DEGs_D        ,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_6_D.results.to_csv('results/ARAUZO_03/enr_tier_6_D.csv', index=False)

enr_tier_6_D.results

In [ ]:
enr_tier_6_E = gp.enrichr(
        gene_list=DEGs_E,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_6_E.results.to_csv('results/ARAUZO_03/enr_tier_6_E.csv', index=False)

enr_tier_6_E.results

In [ ]:
enr_tier_6_F = gp.enrichr(
        gene_list=DEGs_F,
        gene_sets="GO_Biological_Process_2025",
        organism="mouse",
    )

enr_tier_6_F.results.to_csv('results/ARAUZO_03/enr_tier_6_F.csv', index=False)

enr_tier_6_F.results

## Save adata

In [ ]:
adata_FAP.write_h5ad('data/ARAUZO_03/processed_adatas/adata_FAP.h5ad')